# Random Forest — Solo datos reales (sin sinteticos)

1433 autenticos originales + 1433 falsos reales (10 y 20 Bs).
Falsos organizados por denominacion/tipo_feature.

In [33]:
import os, random
import cv2, numpy as np
from pathlib import Path
from collections import Counter

random.seed(42); np.random.seed(42)
print("OpenCV:", cv2.__version__)

OpenCV: 4.13.0


## 2. Alineador + Extractor

In [34]:
TEMPLATE_SIZE = (100, 100)
HIST_BINS, LAPLACIAN_BINS = 32, 32
ORB_FEATURES = 2000
RANSAC_THRESHOLD = 5.0
MATCH_RATIO = 0.75

class Aligner:
    def __init__(self, d):
        self.t={}; self.tk={}; self.td={}
        self.orb=cv2.ORB_create(nfeatures=ORB_FEATURES,scaleFactor=1.2,nlevels=1,edgeThreshold=3)
        ip=dict(algorithm=6,table_number=12,key_size=20,multi_probe_level=2)
        sp=dict(checks=100)
        self.flann=cv2.FlannBasedMatcher(ip,sp)
        for f in sorted(Path(d).glob("*.png")):
            ft=f.stem; img=cv2.imread(str(f),cv2.IMREAD_GRAYSCALE)
            if img is None: continue
            img=cv2.resize(img,TEMPLATE_SIZE); self.t[ft]=img
            kp,des=self.orb.detectAndCompute(img,None)
            self.tk[ft]=kp if kp else []; self.td[ft]=des
    def align(self,img,ft):
        if ft not in self.t or img is None: return None
        ikp,ides=self.orb.detectAndCompute(img,None)
        if ides is not None and len(ides)>=4:
            try:
                td=self.td.get(ft)
                if td is not None and len(td)>=4:
                    m=self.flann.knnMatch(td,ides,k=2)
                    g=[a for a,b in m if a.distance<MATCH_RATIO*b.distance]
                    if len(g)>=4:
                        sp=np.float32([self.tk[ft][a.queryIdx].pt for a in g]).reshape(-1,1,2)
                        dp=np.float32([ikp[a.trainIdx].pt for a in g]).reshape(-1,1,2)
                        H,_=cv2.findHomography(dp,sp,cv2.RANSAC,RANSAC_THRESHOLD)
                        if H is not None:
                            return cv2.warpPerspective(img,H,TEMPLATE_SIZE,borderMode=cv2.BORDER_CONSTANT,borderValue=0)
            except: pass
        return cv2.resize(img,TEMPLATE_SIZE)

def extract_features(img):
    h=cv2.calcHist([img],[0],None,[HIST_BINS],[0,256]); h=cv2.normalize(h,h).flatten()
    l=cv2.Laplacian(img,cv2.CV_64F); l=np.abs(l).astype(np.uint8)
    lh=cv2.calcHist([l],[0],None,[LAPLACIAN_BINS],[0,256]); lh=cv2.normalize(lh,lh).flatten()
    return np.concatenate([h,lh]).astype(np.float32)

aligner = Aligner("../rpi_deployment/templates")
print(f"Aligner: {len(aligner.t)} templates")

Aligner: 21 templates


## 3. AUTENTICOS (originales, sin aumentacion)

In [32]:
auth_dir = Path("../rpi_deployment/training_data/autenticos")
auth_files = list(auth_dir.glob("*.png"))
random.shuffle(auth_files)

X_auth = []; by_class = Counter()
for f in auth_files:
    ft = None
    for p in ["animal_","ir_","personaje_","serie_a","valor_"]:
        if f.stem.startswith(p):
            if p=="serie_a": ft="serie_a"
            else:
                end=f.stem.find('_',len(p))
                ft=f.stem[:end] if end!=-1 else p.rstrip('_')
            break
    if ft is None: continue
    if by_class[ft] >= 500: continue
    by_class[ft] += 1
    img=cv2.imread(str(f),cv2.IMREAD_GRAYSCALE)
    if img is None: continue
    a=aligner.align(img,ft)
    if a is None: continue
    X_auth.append(extract_features(a))

for ft,n in sorted(by_class.items()): print(f"  {ft}: {n}")
print(f"Total autenticos: {len(X_auth)}")

  animal_100bs: 289
  animal_10bs: 291
  animal_200bs: 291
  animal_20bs: 288
  animal_50bs: 287
  ir_100bs: 500
  ir_10bs: 500
  ir_200bs: 500
  ir_20bs: 500
  ir_50bs: 500
  personaje_100bs: 329
  personaje_10bs: 286
  personaje_200bs: 288
  personaje_20bs: 290
  personaje_50bs: 290
  serie_a: 500
  valor_100bs: 291
  valor_10bs: 288
  valor_200bs: 286
  valor_20bs: 284
  valor_50bs: 293
Total autenticos: 7371


## 4. FALSOS REALES (de la K210, mezclados)

In [35]:
fakes_dir = Path("../Data/raw/fakes")
fake_files = list(fakes_dir.rglob("*.jpg"))
random.shuffle(fake_files)
print(f"Falsos disponibles: {len(fake_files)}")

X_fake = []; fake_counts = Counter()
for f in fake_files:
    img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
    if img is None: continue
    # El tipo de feature = nombre de la carpeta padre
    # Ej: fakes/10/personaje_10bs/img.jpg -> ft = "personaje_10bs"
    ft = f.parent.name
    if ft not in aligner.t:
        continue
    a = aligner.align(img, ft)
    if a is None: continue
    X_fake.append(extract_features(a))
    fake_counts[ft] += 1

for ft, n in sorted(fake_counts.items()):
    print(f"  {ft}: {n}")
print(f"Total falsos: {len(X_fake)}")

Falsos disponibles: 1433


Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment


  animal_10bs: 72
  animal_20bs: 123
  ir_10bs: 111
  ir_20bs: 242
  personaje_10bs: 39
  personaje_20bs: 249
  serie_a: 310
  valor_10bs: 76
  valor_20bs: 211
Total falsos: 1433


Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment
Corrupt JPEG data: premature end of data segment


## 5. Balancear + Split

In [36]:
n = min(len(X_auth), len(X_fake))
X_auth, X_fake = X_auth[:n], X_fake[:n]
X = np.vstack([X_auth, X_fake]).astype(np.float32)
y = np.hstack([np.ones(n,dtype=np.int32), np.zeros(n,dtype=np.int32)])

idx = np.random.permutation(len(X))
split = int(0.8 * len(X))
X_train, y_train = X[idx[:split]], y[idx[:split]]
X_test, y_test = X[idx[split:]], y[idx[split:]]

print(f"Total: {len(X)} ({n} auth + {n} fake)")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Total: 2866 (1433 auth + 1433 fake)
Train: 2292, Test: 574


## 6. Entrenar Random Forest

In [37]:
rf = cv2.ml.RTrees_create()
rf.setMaxDepth(15); rf.setMinSampleCount(2)
rf.setTermCriteria((cv2.TERM_CRITERIA_MAX_ITER + cv2.TERM_CRITERIA_EPS, 150, 0.01))
rf.train(cv2.ml.TrainData_create(X_train, cv2.ml.ROW_SAMPLE, y_train))
print("Entrenado!")

Entrenado!


## 7. Evaluar

In [38]:
y_pred = np.array([int(rf.predict(r.reshape(1,-1))[1][0]) for r in X_test])
acc = np.mean(y_pred == y_test)
tp = sum((y_pred==1)&(y_test==1)); fp = sum((y_pred==1)&(y_test==0))
tn = sum((y_pred==0)&(y_test==0)); fn = sum((y_pred==0)&(y_test==1))
prec = tp/(tp+fp); rec = tp/(tp+fn)
f1 = 2*prec*rec/(prec+rec)
print(f"Accuracy:  {acc*100:.2f}%")
print(f"Precision: {prec*100:.2f}%")
print(f"Recall:    {rec*100:.2f}%")
print(f"F1:        {f1*100:.2f}%")
print(f"\nMatriz: TP={tp} FP={fp} TN={tn} FN={fn}")

Accuracy:  99.48%
Precision: 99.64%
Recall:    99.29%
F1:        99.46%

Matriz: TP=278 FP=1 TN=293 FN=2


## 8. Exportar

In [39]:
rf.save("../rpi_deployment/rf_classifier.xml")
print("rf_classifier.xml guardado")

rf_classifier.xml guardado
